# Genie Cache & Queue — Demo

A API do Genie tem um hard limit de **5 chamadas por minuto por workspace**.
Este notebook demonstra o problema e como o app resolve com cache semantico, retry e fila.

## Cenarios:
1. **Sem o app**: 7 chamadas diretas ao Genie → ultimas recebem **HTTP 429**
2. **Com o app (1a rodada)**: Mesmas 7 queries via app → todas completam (retry automatico)
3. **Com o app (2a rodada)**: Mesmas queries de novo → **todas do cache em <0.1s**

O app e um **drop-in replacement** — mesmos endpoints, mesmos payloads, mesmas respostas.
Basta trocar a URL base.

## Setup

In [ ]:
import requests
import time
import json

WORKSPACE_HOST = "https://fevm-genie-cache-test.cloud.databricks.com"
APP_HOST = "https://genie-cache-queue-7474650836156271.aws.databricksapps.com"
SPACE_ID = "01f11f1ae00114379e7671c8a4b8459f"
WAREHOUSE_ID = "20cbd6750a2ef9ce"

try:
    TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
except:
    import os
    TOKEN = os.getenv("DATABRICKS_TOKEN", "")

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}

questions = [
    "How many customers are there?",
    "What is the total revenue by region?",
    "Top 5 suppliers by order count",
    "Average order value per year",
    "How many parts are in the catalog?",
    "Total revenue for EUROPE region",
    "Number of orders in 1995",
]

def extract_genie_response(data):
    """Extract SQL and text from a Genie API response (works for both direct and app)."""
    sql = None
    text = None
    attachments = data.get("attachments", [])
    for att in attachments:
        if isinstance(att, dict):
            if "query" in att and att["query"]:
                sql = att["query"].get("query", "")
            if "text" in att and att["text"]:
                text = att["text"].get("content", "")
    # Fallback: sql_query at root (app adds this)
    if not sql:
        sql = data.get("sql_query", "")
    return sql, text

print(f"Workspace: {WORKSPACE_HOST}")
print(f"App:       {APP_HOST}")
print(f"Space:     {SPACE_ID}")
print(f"Token:     {'***' + TOKEN[-4:] if TOKEN else 'NOT SET'}")
print(f"Questions: {len(questions)}")

## Cenario 1: Chamadas diretas ao Genie (sem app)

7 queries diretas na API do Genie. Limite: **5/min por workspace**.
As chamadas 6 e 7 devem retornar **HTTP 429 (Too Many Requests)**.

In [ ]:
print("=" * 80)
print("CENARIO 1: Chamadas DIRETAS ao Genie (sem cache/retry)")
print("=" * 80)
print()

direct_results = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{WORKSPACE_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers,
        json={"content": q},
        timeout=120,
    )
    elapsed = time.time() - start
    http_status = resp.status_code

    if http_status == 429:
        retry_after = resp.headers.get("Retry-After", "?")
        print(f"[{i}/7] RATE LIMITED (429) | {elapsed:.1f}s | Retry-After: {retry_after}s")
        print(f"       Query: {q}")
        print(f"       Resultado: BLOQUEADO pelo rate limit")
        direct_results.append({"q": q, "http": 429, "time": elapsed, "sql": None, "text": None})
    elif http_status == 200:
        data = resp.json()
        sql, text = extract_genie_response(data)
        conv_id = data.get("conversation_id", "")[:20]
        print(f"[{i}/7] OK (200) | {elapsed:.1f}s | conv={conv_id}...")
        print(f"       Query: {q}")
        if sql:
            print(f"       SQL:   {sql[:100]}")
        if text:
            print(f"       Resp:  {text[:120]}")
        direct_results.append({"q": q, "http": 200, "time": elapsed, "sql": sql, "text": text})
    else:
        print(f"[{i}/7] ERROR ({http_status}) | {elapsed:.1f}s")
        print(f"       Query: {q}")
        print(f"       Erro:  {resp.text[:120]}")
        direct_results.append({"q": q, "http": http_status, "time": elapsed, "sql": None, "text": None})
    print()

ok = sum(1 for r in direct_results if r["http"] == 200)
blocked = sum(1 for r in direct_results if r["http"] == 429)
print(f"RESULTADO: {ok} OK, {blocked} bloqueadas por rate limit (429)")

## Cenario 2: Via App (cache + retry + fila)

Mesmas 7 queries, mas agora via app. O endpoint e **identico** ao Genie real:
```
POST /api/2.0/genie/spaces/{space_id}/start-conversation
{"content": "pergunta"}
```
So muda a URL base.

### Configurar o app

In [ ]:
# Configurar space_id e warehouse_id via API
config_resp = requests.put(
    f"{APP_HOST}/api/v1/config",
    headers=headers,
    json={
        "genie_space_id": SPACE_ID,
        "sql_warehouse_id": WAREHOUSE_ID,
        "similarity_threshold": 0.92,
        "cache_ttl_seconds": 86400,
    },
)
print("Config:")
print(json.dumps(config_resp.json(), indent=2))

### 2a. Primeira rodada (cache miss — popula cache)

Queries novas. O app chama Genie real, gerencia rate limit, enfileira se necessario.
**O caller nunca ve 429.**

In [ ]:
print("=" * 80)
print("CENARIO 2a: Via APP — primeira rodada (popula cache)")
print("=" * 80)
print()

app_results_1 = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{APP_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers,
        json={"content": q},
        timeout=180,
    )
    elapsed = time.time() - start

    if resp.status_code != 200:
        print(f"[{i}/7] HTTP {resp.status_code} | {elapsed:.1f}s | {q}")
        print(f"       Erro: {resp.text[:150]}")
        app_results_1.append({"q": q, "time": elapsed, "from_cache": False, "sql": None, "text": None, "status": "ERROR"})
        print()
        continue

    data = resp.json()
    status = data.get("status", "UNKNOWN")
    conv_id = data.get("conversation_id", "")
    from_cache = conv_id.startswith("ccache_")
    sql, text = extract_genie_response(data)

    source = "CACHE" if from_cache else "GENIE"
    print(f"[{i}/7] {status} | {source} | {elapsed:.1f}s")
    print(f"       Query: {q}")
    if sql:
        print(f"       SQL:   {sql[:100]}")
    if text:
        print(f"       Resp:  {text[:120]}")
    print()

    app_results_1.append({"q": q, "time": elapsed, "from_cache": from_cache, "sql": sql, "text": text, "status": status})

cache_hits = sum(1 for r in app_results_1 if r["from_cache"])
genie_calls = sum(1 for r in app_results_1 if not r["from_cache"] and r["status"] != "ERROR")
failed = sum(1 for r in app_results_1 if r["status"] in ("FAILED", "ERROR"))
print(f"RESULTADO: {genie_calls} Genie calls, {cache_hits} cache hits, {failed} erros")

### 2b. Segunda rodada (tudo do cache)

Mesmas 7 queries. Todas ja estao cacheadas — resposta instantanea, zero chamadas ao Genie.
**O SQL retornado e identico ao da primeira rodada.**

In [ ]:
print("=" * 80)
print("CENARIO 2b: Via APP — segunda rodada (tudo do cache)")
print("=" * 80)
print()

app_results_2 = []
for i, q in enumerate(questions, 1):
    start = time.time()
    resp = requests.post(
        f"{APP_HOST}/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
        headers=headers,
        json={"content": q},
        timeout=120,
    )
    elapsed = time.time() - start

    data = resp.json() if resp.status_code == 200 else {}
    conv_id = data.get("conversation_id", "")
    from_cache = conv_id.startswith("ccache_")
    sql, text = extract_genie_response(data)

    # Compare SQL with 1st round
    prev_sql = app_results_1[i-1]["sql"] if i-1 < len(app_results_1) else None
    sql_match = "MATCH" if sql and prev_sql and sql.strip() == prev_sql.strip() else "NEW" if sql else "-"

    print(f"[{i}/7] CACHE HIT | {elapsed:.3f}s | SQL {sql_match}")
    print(f"       Query: {q}")
    if sql:
        print(f"       SQL:   {sql[:100]}")
    print()

    app_results_2.append({"q": q, "time": elapsed, "from_cache": from_cache, "sql": sql})

total_time = sum(r["time"] for r in app_results_2)
all_cached = all(r["from_cache"] for r in app_results_2)
print(f"RESULTADO: {len(app_results_2)}/{len(questions)} cache hits | Total: {total_time:.2f}s | Media: {total_time/len(questions):.3f}s/query")
print(f"Todas do cache: {all_cached}")

### Cache via API

In [ ]:
cache_resp = requests.get(f"{APP_HOST}/api/v1/cache", headers=headers)
cache_entries = cache_resp.json() if cache_resp.status_code == 200 else []
print(f"Entradas no cache: {len(cache_entries)}\n")
for e in cache_entries:
    print(f"  Query:  {e['query_text'][:60]}")
    print(f"  SQL:    {e['sql_query'][:80]}")
    print(f"  Uses:   {e['use_count']}  |  Created: {e['created_at']}")
    print()

## Comparacao Final

Lado a lado: chamada direta vs app (1a rodada) vs app (2a rodada).

In [ ]:
print("=" * 90)
print("COMPARACAO")
print("=" * 90)
print(f"{'Query':30s} | {'Direto':>14s} | {'App 1a rodada':>16s} | {'App 2a rodada':>14s}")
print("-" * 90)

for i, q in enumerate(questions):
    d = direct_results[i] if i < len(direct_results) else {"http": "?", "time": 0}
    a1 = app_results_1[i] if i < len(app_results_1) else {"status": "?", "time": 0, "from_cache": False}
    a2 = app_results_2[i] if i < len(app_results_2) else {"time": 0, "from_cache": False}

    if d["http"] == 429:
        d_col = "429 BLOCKED"
    elif d["http"] == 200:
        d_col = f"{d['time']:.1f}s OK"
    else:
        d_col = f"{d['http']} ERR"

    if a1.get("from_cache"):
        a1_col = f"{a1['time']:.1f}s CACHE"
    elif a1.get("status") == "COMPLETED":
        a1_col = f"{a1['time']:.1f}s GENIE"
    else:
        a1_col = f"{a1['time']:.1f}s {a1.get('status','?')[:6]}"

    a2_col = f"{a2['time']:.2f}s CACHE"

    print(f"  {q[:28]:28s} | {d_col:>14s} | {a1_col:>16s} | {a2_col:>14s}")

print()
print("Legenda:")
print("  Direto:  chamada direta ao Genie — recebe 429 apos 5 chamadas")
print("  App 1a:  via app, cache vazio — chama Genie com retry automatico")
print("  App 2a:  via app, cache cheio — retorno instantaneo, mesmo SQL")